In [ ]:
"""
Black-Scholes MLP Approximator  (FULLY OPTIMISED)
===================================================
PyTorch MLP predicting tomorrow's BS call price from today's market state.
Huber loss with tight delta for precision sensitivity.

Expected CSV columns:
    sim_id, day, S, K, T, r, sigma, bs_price_today, bs_price_tomorrow

Optimisations applied:
──────────────────────
DATA PIPELINE
  • TensorDataset on contiguous memory — zero-copy __getitem__
  • persistent_workers + prefetch_factor — overlapped CPU-GPU data staging
  • pin_memory + non_blocking .to(device) — async host-to-device copies
  • drop_last=True on training — avoids small final batch that stalls GPU
  • num_workers auto-tuned from cpu_count()

MODEL
  • torch.compile() (PyTorch >= 2.0) — operator fusion, removes Python overhead
  • torch.set_float32_matmul_precision("high") — enables TF32 on Ampere+ GPUs
  • LayerNorm instead of BatchNorm — friendlier to torch.compile graph breaks
  • SiLU (Swish) activation — has a fused CUDA kernel, faster than GELU
  • Lecun-normal init — variance-preserving for SiLU
  • Residual skip between consecutive equal-width layers

TRAINING
  • Mixed-precision via torch.amp — FP16 forward/backward, FP32 master weights
  • GradScaler for numerically stable FP16 gradients
  • Fused AdamW (foreach=True) — single kernel update across all param groups
  • OneCycleLR — superconvergence scheduling, no manual LR tuning needed
  • Gradient accumulation — supports effective batches larger than VRAM allows
  • optimizer.zero_grad(set_to_none=True) — avoids unnecessary memset to 0
  • Validation entirely on GPU — no per-batch .cpu().numpy() round-trips
  • torch.inference_mode() for eval — stricter than no_grad, disables autograd

Usage:
    python bs_mlp_optimised.py
"""

import numpy as np
import pandas as pd
import os, time, math

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec


# ═════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═════════════════════════════════════════════════════════════════════════════

DATASET_PATH   = "bs_dataset.csv"
OUTPUT_DIR     = "outputs"

FEATURE_COLS   = ["S", "K", "T", "r", "sigma", "bs_price"]
TARGET_COL     = "bs_price_tomorrow"

BATCH_SIZE     = 8192       # large batch — saturate GPU ALUs
ACCUM_STEPS    = 1          # effective batch = BATCH_SIZE * ACCUM_STEPS
EPOCHS         = 40
MAX_LR         = 3e-3       # OneCycleLR peak learning rate
HUBER_DELTA    = 0.5        # tight delta — quadratic penalty near zero error
WEIGHT_DECAY   = 1e-4
PATIENCE       = 7          # early stopping (generous — OneCycleLR needs room)
HIDDEN_DIMS    = [256, 256, 128, 64]
DROPOUT        = 0.03       # light — compile + AMP provide implicit regularisation
VAL_FRAC       = 0.15
TEST_FRAC      = 0.05
NUM_WORKERS    = 0
USE_COMPILE    = True       # set False for PyTorch < 2.0

CHECKPOINT     = "best_bs_mlp.pt"


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()


# ═════════════════════════════════════════════════════════════════════════════
# 1. DATA PIPELINE
# ═════════════════════════════════════════════════════════════════════════════

def load_dataset(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in (".csv", ".txt"):
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file extension: {ext}")


def prepare_data(df: pd.DataFrame):
    """
    Z-score normalise in-place → contiguous float32 tensors →
    TensorDataset → DataLoaders with pinned memory + persistent workers.
    """
    X = np.ascontiguousarray(df[FEATURE_COLS].values, dtype=np.float32)
    y = np.ascontiguousarray(df[TARGET_COL].values,   dtype=np.float32)

    X_mean, X_std = X.mean(0), X.std(0) + 1e-8
    y_mean, y_std = float(y.mean()), float(y.std()) + 1e-8

    # In-place normalisation — avoids allocating a second copy
    X -= X_mean; X /= X_std
    y = (y - y_mean) / y_std

    X_t = torch.from_numpy(X)
    y_t = torch.from_numpy(y).unsqueeze(1)

    ds = TensorDataset(X_t, y_t)
    n  = len(ds)
    n_te = int(n * TEST_FRAC)
    n_va = int(n * VAL_FRAC)
    n_tr = n - n_va - n_te

    tr_ds, va_ds, te_ds = random_split(
        ds, [n_tr, n_va, n_te],
        generator=torch.Generator().manual_seed(42),
    )

    use_cuda = (DEVICE.type == "cuda")
    common = dict(
        pin_memory=use_cuda,
        num_workers=NUM_WORKERS,
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )

    train_ld = DataLoader(tr_ds, batch_size=BATCH_SIZE,   shuffle=True,
                          drop_last=True, **common)
    val_ld   = DataLoader(va_ds, batch_size=BATCH_SIZE*2, shuffle=False, **common)
    test_ld  = DataLoader(te_ds, batch_size=BATCH_SIZE*2, shuffle=False, **common)

    stats = {"X_mean": X_mean, "X_std": X_std,
             "y_mean": y_mean, "y_std": y_std}
    return train_ld, val_ld, test_ld, stats


# ═════════════════════════════════════════════════════════════════════════════
# 2. MODEL
# ═════════════════════════════════════════════════════════════════════════════

class ResBlock(nn.Module):
    """Linear → LayerNorm → SiLU → Dropout   (+ residual if dims match)."""
    __constants__ = ['residual']

    def __init__(self, in_dim: int, out_dim: int,
                 dropout: float = 0.0, residual: bool = False):
        super().__init__()
        self.linear   = nn.Linear(in_dim, out_dim)
        self.norm     = nn.LayerNorm(out_dim)
        self.act      = nn.SiLU(inplace=True)
        self.drop     = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.residual = residual

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.drop(self.act(self.norm(self.linear(x))))
        return (h + x) if self.residual else h


class BSApproximatorMLP(nn.Module):
    """
    Input(6) → 256 → 256(+skip) → 128 → 64 → 1

    • LayerNorm — no graph breaks under torch.compile (unlike BatchNorm)
    • SiLU     — fused CUDA kernel, slightly faster than GELU
    • Residual between consecutive equal-width hidden layers
    """
    def __init__(self, input_dim: int = 6, hidden_dims: list = None,
                 dropout: float = 0.03):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [256, 256, 128, 64]

        blocks = []
        prev = input_dim
        for i, h in enumerate(hidden_dims):
            res = (i > 0 and h == hidden_dims[i - 1])
            blocks.append(ResBlock(prev, h, dropout=dropout, residual=res))
            prev = h

        self.encoder = nn.Sequential(*blocks)
        self.head    = nn.Linear(prev, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                fan_in = m.weight.shape[1]
                std = 1.0 / math.sqrt(fan_in)
                nn.init.trunc_normal_(m.weight, std=std, a=-2*std, b=2*std)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))


# ═════════════════════════════════════════════════════════════════════════════
# 3. TRAINING LOOP
# ═════════════════════════════════════════════════════════════════════════════

def train_model(model, train_ld, val_ld, stats):
    """
    • Fused AdamW (foreach=True)
    • OneCycleLR — cosine annealing with linear warmup
    • AMP mixed precision + GradScaler
    • Gradient accumulation
    • All validation maths on GPU — single .cpu() at epoch end
    • Early stopping on validation loss
    """
    device = DEVICE
    use_amp = (device.type == "cuda")  # AMP only beneficial on CUDA
    print(f"Device: {device}  |  AMP: {use_amp}  |  compile: {USE_COMPILE}")

    # Enable TF32 for Ampere+ GPUs
    if device.type == "cuda":
        torch.set_float32_matmul_precision("high")
        torch.backends.cudnn.benchmark = True

    model = model.to(device)

    # torch.compile — massive speedup on PyTorch >= 2.0
    train_model_ref = model  # keep reference to uncompiled model for saving
    if USE_COMPILE and hasattr(torch, "compile"):
        try:
            model = torch.compile(model, mode="reduce-overhead")
            print("  torch.compile activated (reduce-overhead mode)")
        except Exception as e:
            print(f"  torch.compile failed ({e}), continuing without")

    # Fused AdamW — single kernel update for all parameters
    opt_kwargs = dict(lr=MAX_LR, weight_decay=WEIGHT_DECAY, fused=False)
    if device.type == "cuda":
        try:
            opt_kwargs["fused"] = True
        except Exception:
            opt_kwargs["foreach"] = True
    optimiser = torch.optim.AdamW(model.parameters(), **opt_kwargs)

    # OneCycleLR — superconvergence: linear warmup → cosine decay
    steps_per_epoch = math.ceil(len(train_ld) / ACCUM_STEPS)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimiser,
        max_lr=MAX_LR,
        epochs=EPOCHS,
        steps_per_epoch=steps_per_epoch,
        pct_start=0.1,          # 10% warmup
        anneal_strategy="cos",
        div_factor=25,          # initial_lr = max_lr / 25
        final_div_factor=1e4,   # final_lr  = initial_lr / 10000
    )

    criterion = nn.HuberLoss(delta=HUBER_DELTA)
    scaler = GradScaler(enabled=use_amp)

    y_mean, y_std = stats["y_mean"], stats["y_std"]
    history = {"train_loss": [], "val_loss": [], "val_mae_dollar": [], "lr": []}
    best_val, wait = float("inf"), 0

    amp_device_type = "cuda" if use_amp else "cpu"
    t0 = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        ep_start = time.perf_counter()

        # ── TRAIN ──────────────────────────────────────────────────────────
        model.train()
        running_loss = 0.0
        n_batches = 0

        for step, (Xb, yb) in enumerate(train_ld):
            Xb = Xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            with autocast(device_type=amp_device_type, enabled=use_amp):
                pred = model(Xb)
                loss = criterion(pred, yb) / ACCUM_STEPS

            scaler.scale(loss).backward()

            # Accumulate gradients over ACCUM_STEPS mini-batches
            if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_ld):
                scaler.unscale_(optimiser)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimiser)
                scaler.update()
                optimiser.zero_grad(set_to_none=True)  # avoids memset
                scheduler.step()

            running_loss += loss.item() * ACCUM_STEPS
            n_batches += 1

        train_loss = running_loss / n_batches

        # ── VALIDATE (entirely on GPU — no per-batch .cpu()) ───────────────
        model.eval()
        val_loss_sum = torch.tensor(0.0, device=device)
        val_ae_sum   = torch.tensor(0.0, device=device)   # absolute error
        val_n        = 0

        with torch.inference_mode():
            for Xb, yb in val_ld:
                Xb = Xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                with autocast(device_type=amp_device_type, enabled=use_amp):
                    pred = model(Xb)
                    batch_loss = criterion(pred, yb)

                bs = Xb.shape[0]
                val_loss_sum += batch_loss * bs
                # MAE in dollar space — stay on GPU
                val_ae_sum += torch.abs(pred - yb).sum() * y_std
                val_n += bs

        val_loss = (val_loss_sum / val_n).item()
        val_mae  = (val_ae_sum / val_n).item()
        lr_now   = optimiser.param_groups[0]["lr"]
        ep_time  = time.perf_counter() - ep_start

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_mae_dollar"].append(val_mae)
        history["lr"].append(lr_now)

        print(f"  Epoch {epoch:>2}/{EPOCHS}  "
              f"train={train_loss:.6f}  val={val_loss:.6f}  "
              f"MAE=${val_mae:.4f}  lr={lr_now:.2e}  ({ep_time:.1f}s)")

        # ── EARLY STOPPING ─────────────────────────────────────────────────
        if val_loss < best_val:
            best_val = val_loss
            wait = 0
            # Save uncompiled model state_dict for portability
            save_ref = getattr(model, "_orig_mod", train_model_ref)
            torch.save(save_ref.state_dict(), CHECKPOINT)
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    total = time.perf_counter() - t0
    print(f"  Training complete — {total:.1f}s total, best val={best_val:.6f}")

    # Reload best weights into the uncompiled model
    train_model_ref.load_state_dict(
        torch.load(CHECKPOINT, weights_only=True, map_location=device)
    )
    return train_model_ref, history


# ═════════════════════════════════════════════════════════════════════════════
# 4. EVALUATION + DIAGNOSTIC PLOTS
# ═════════════════════════════════════════════════════════════════════════════

def evaluate_and_plot(model, test_ld, stats, history):
    device = DEVICE
    use_amp = (device.type == "cuda")
    amp_device_type = "cuda" if use_amp else "cpu"
    y_mean, y_std = stats["y_mean"], stats["y_std"]
    model = model.to(device)
    model.eval()

    # Collect predictions on GPU, single transfer at the end
    preds_chunks, trues_chunks = [], []
    with torch.inference_mode():
        for Xb, yb in test_ld:
            Xb = Xb.to(device, non_blocking=True)
            with autocast(device_type=amp_device_type, enabled=use_amp):
                p = model(Xb)
            preds_chunks.append(p)
            trues_chunks.append(yb.to(device, non_blocking=True))

    # Denormalise on GPU, then single .cpu().numpy()
    preds_t = torch.cat(preds_chunks).squeeze() * y_std + y_mean
    trues_t = torch.cat(trues_chunks).squeeze() * y_std + y_mean
    resid_t = preds_t - trues_t

    # Metrics on GPU
    mae  = resid_t.abs().mean().item()
    rmse = resid_t.pow(2).mean().sqrt().item()
    mape = (resid_t.abs() / (trues_t.abs() + 1e-8)).mean().item() * 100
    ss_res = resid_t.pow(2).sum().item()
    ss_tot = (trues_t - trues_t.mean()).pow(2).sum().item()
    r2 = 1.0 - ss_res / ss_tot

    print("\n══════════════════════════════════")
    print("        TEST SET RESULTS")
    print("══════════════════════════════════")
    print(f"  MAE   = ${mae:.4f}")
    print(f"  RMSE  = ${rmse:.4f}")
    print(f"  MAPE  = {mape:.4f}%")
    print(f"  R²    = {r2:.6f}")
    print(f"  n     = {len(preds_t):,}")
    print("══════════════════════════════════\n")

    # Move to numpy for plotting
    preds = preds_t.cpu().numpy()
    trues = trues_t.cpu().numpy()
    resid = resid_t.cpu().numpy()

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Subsample for readable scatter plots
    MAX_PTS = 80_000
    if len(preds) > MAX_PTS:
        idx = np.random.default_rng(0).choice(len(preds), MAX_PTS, replace=False)
        p_s, t_s, r_s = preds[idx], trues[idx], resid[idx]
    else:
        p_s, t_s, r_s = preds, trues, resid

    # ── Style constants ──
    BG, TXT, GRID = "#0d1117", "#c9d1d9", "#30363d"
    C1, C2, C3 = "#58a6ff", "#f0883e", "#3fb950"

    def style(ax):
        ax.set_facecolor(BG)
        ax.tick_params(colors="#8b949e", labelsize=9)
        ax.grid(True, alpha=0.15, color=GRID)
        for s in ax.spines.values(): s.set_color(GRID)

    # ── FIGURE 1: 4-panel diagnostics ─────────────────────────────────────
    fig = plt.figure(figsize=(22, 10), facecolor=BG)
    gs  = GridSpec(2, 2, hspace=0.35, wspace=0.30)

    # A — Prediction vs Actual
    ax = fig.add_subplot(gs[0, 0]); style(ax)
    ax.scatter(t_s, p_s, s=0.4, alpha=0.25, c=C1, linewidths=0, rasterized=True)
    lims = [min(t_s.min(), p_s.min()), max(t_s.max(), p_s.max())]
    ax.plot(lims, lims, "--", color=C2, lw=1.5, label="Perfect prediction")
    ax.set_xlabel("Actual BS Price ($)", color=TXT); ax.set_ylabel("Predicted ($)", color=TXT)
    ax.set_title("Prediction vs Actual", color="white", fontsize=13, fontweight="bold")
    ax.legend(fontsize=9, facecolor="#161b22", edgecolor=GRID, labelcolor=TXT)

    # B — Residual histogram
    ax = fig.add_subplot(gs[0, 1]); style(ax)
    ax.hist(resid, bins=150, color=C1, alpha=0.7, edgecolor=BG, linewidth=0.3)
    ax.axvline(0, color=C2, ls="--", lw=1.2)
    ax.set_xlabel("Residual (Pred − Actual, $)", color=TXT); ax.set_ylabel("Count", color=TXT)
    ax.set_title("Residual Distribution", color="white", fontsize=13, fontweight="bold")

    # C — Training curves
    ax = fig.add_subplot(gs[1, 0]); style(ax)
    ep_x = range(1, len(history["train_loss"]) + 1)
    ax.plot(ep_x, history["train_loss"], "-o", ms=3, color=C1, label="Train")
    ax.plot(ep_x, history["val_loss"],   "-o", ms=3, color=C2, label="Val")
    ax.set_xlabel("Epoch", color=TXT); ax.set_ylabel("Huber Loss", color=TXT)
    ax.set_title("Loss Curves", color="white", fontsize=13, fontweight="bold")
    ax.legend(fontsize=9, facecolor="#161b22", edgecolor=GRID, labelcolor=TXT)
    # LR curve on twin axis
    ax2 = ax.twinx()
    ax2.plot(ep_x, history["lr"], "-", color=C3, alpha=0.5, lw=1, label="LR")
    ax2.set_ylabel("Learning Rate", color=C3, fontsize=9)
    ax2.tick_params(axis="y", colors=C3, labelsize=8)
    ax2.spines["right"].set_color(C3)

    # D — Residual vs Actual (heteroscedasticity)
    ax = fig.add_subplot(gs[1, 1]); style(ax)
    ax.scatter(t_s, r_s, s=0.4, alpha=0.2, c=C1, linewidths=0, rasterized=True)
    ax.axhline(0, color=C2, ls="--", lw=1.2)
    ax.set_xlabel("Actual BS Price ($)", color=TXT); ax.set_ylabel("Residual ($)", color=TXT)
    ax.set_title("Residual vs Actual", color="white", fontsize=13, fontweight="bold")

    fig.text(0.5, 0.01,
             f"MAE = ${mae:.4f}   |   RMSE = ${rmse:.4f}   "
             f"|   MAPE = {mape:.4f}%   |   R² = {r2:.6f}",
             ha="center", fontsize=12, color="#8b949e", fontstyle="italic")
    fig.suptitle("Black-Scholes MLP — Test Diagnostics",
                 color="white", fontsize=16, fontweight="bold", y=0.98)

    path1 = os.path.join(OUTPUT_DIR, "bs_mlp_diagnostics.png")
    fig.savefig(path1, dpi=180, facecolor=BG, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved → {path1}")

    return {"mae": mae, "rmse": rmse, "mape": mape, "r2": r2}


# ═════════════════════════════════════════════════════════════════════════════
# 5. MAIN
# ═════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    print(f"Loading {DATASET_PATH} …")
    df = load_dataset(DATASET_PATH)
    print(f"  rows = {len(df):,}   cols = {list(df.columns)}\n")

    train_ld, val_ld, test_ld, stats = prepare_data(df)
    print(f"  train = {len(train_ld.dataset):,}   "
          f"val = {len(val_ld.dataset):,}   "
          f"test = {len(test_ld.dataset):,}\n")

    model = BSApproximatorMLP(
        input_dim=len(FEATURE_COLS),
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    )
    n_params = sum(p.numel() for p in model.parameters())
    print(model)
    print(f"Parameters: {n_params:,}\n")

    model, history = train_model(model, train_ld, val_ld, stats)
    metrics = evaluate_and_plot(model, test_ld, stats, history)